<div class="alert alert-success">

# Homework 9B: Multiple Linear Regression, Gradients (Practical)

### EECS 245, Winter 2026 at the University of Michigan
    
</div>

### Instructions (new!)

Homework 9B is contained **entirely** in this Jupyter Notebook. Some problems ask you to write code, as per usual. These problems are marked [Autograded 💻]. Other problems ask you to write explanations **right here in this notebook**; these problems are marked [Manually Graded ✏️]. To submit Homework 9B, all you need to do is submit this notebook to the **Homework 9B** assignment on Gradescope. There, we will both autograde your code and read and grade your English explanations.

To access this notebook:

1. **Set up a Jupyter Notebook environment locally, and use `git` to clone our course repository (preferred).** For instructions on how to do this, see the [**Environment Setup**](https://eecs245.org/env-setup) page of the course website.
1. **Use the EECS 245 DataHub.** To do this, click the link provided on the course website under Homework 9B.

Final note: the autograded questions in this notebook have no hidden tests: the tests you see here in your notebook are the same tests that will be run on your submission on Gradescope. When you submit to Gradescope, you'll see your autograder score shortly, and will see your score on [Manually Graded ✏️] questions once our graders read them, approximately a week after the deadline.

In [ ]:
# Run this cell.
import numpy as np
import pandas as pd
import time

pd.options.plotting.backend = "plotly"

import plotly.express as px
import plotly.io as pio
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

# Set default layout for all plotly figures
import plotly.graph_objs as go

custom_template = go.layout.Template(pio.templates["plotly_white"])
custom_template.layout.plot_bgcolor = "white"
custom_template.layout.paper_bgcolor = "white"
custom_template.layout.margin = dict(l=60, r=60, t=60, b=60)
custom_template.layout.width = 700
custom_template.layout.font = dict(
    family="Palatino Linotype, Palatino, serif",
    color="black"
)

pio.templates["custom"] = custom_template
pio.templates.default = "custom"

import hw09_util as util

---

## Problem 1: Home Run! (11 pts)

In this problem, we'll work with a real dataset that describes the number of home runs in the MLB per year. Our goal will be to fit various models that predict the number of home runs, $y_i$, given the year, $x_i$. 

Run the cells below to load in and visualize the data.

In [ ]:
homeruns = pd.read_csv('data/homeruns.csv')
homeruns

In [ ]:
fig = homeruns.plot(kind='scatter', x='Year', y='Homeruns', title='Homeruns vs. Year')
fig.show(renderer='notebook')

In parts **a)** through **c)**, you are given a model, $h(x_i)$. For each one, you should implement the function `create_and_train_<problem_part>`, which:

1. Implements the necessary data transformations and use `sklearn`'s `LinearRegression` class to fit the model. 
1. Displays, in this notebook, a scatter plot of the model's predictions, when `show=True`.
1. Returns a **tuple** containing three values:
    1. The model's optimal parameters, as a `numpy` array.
    1. The root mean squared error (RMSE) of the model's predictions.
    1. The $R^2$ score of the model's predictions.

Root mean squared error is the square root of mean squared error; its values are in the same units as $y$. The $R^2$ score of a model's predictions is equal to $\text{correlation}(\text{actual y values}, \text{predicted y values})^2$.

There are helper functions in this section to help you with all steps; start by understanding how they work. Part **a)** has been done for you as an example. Most of your work is in implementing the feature transformations. While it's not required, it's a good idea to be able to take the model's optimal parameters and use them to write down a formula on paper for the actual model.

### Problem 1a) (0 pts) [Autograded 💻]

$$h(x_i) = w_0 + w_1 x_i + w_2 x_i^2$$

In [ ]:
from sklearn.linear_model import LinearRegression # Only needs to be imported once.
from sklearn.metrics import mean_squared_error
# Helper function, feel free to reuse!
def plot_raw_data_and_predictions(X_raw, X_proc, y, model):
    """
    Inputs:
        - X_raw, a DataFrame with a single column, `Year`
        - X_proc, a DataFrame that results from adding new columns to X_raw corresponding to the new features
        - y, a Series with `Homeruns` values
        - model, an **already fit** `LinearRegression` object, fit on (X_proc, y)
    Outputs:
        - Returns a plotly figure object showing a scatter plot of the raw data in blue, with the model's predictions overlaid in orange

    """
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=X_raw.to_numpy().flatten(),
            y=y,
            mode='markers',
            name='Actual'
        )
    )
    fig.add_trace(
        go.Scatter(
            x=X_raw.to_numpy().flatten(),
            y=model.predict(X_proc),
            mode='lines',
            name='Predictions',
            line=dict(color='orange', width=4)
        )
    )
    fig.update_layout(
        title='Homeruns vs. Year (with Model Predictions)',
        xaxis_title='Year',
        yaxis_title='Homeruns'
    )
    return fig

# Helper function, feel free to reuse!
def create_and_train_helper(X_raw, X_proc, y, show=True):
    '''
    Inputs:
        - X_raw, a DataFrame with a single column, `Year`
        - X_proc, a DataFrame that results from adding new columns to X_raw corresponding to the new features
        - y, a Series with `Homeruns` values
    Outputs:
        - Returns a tuple containing the model's optimal parameters, the root mean squared error of the model's predictions, and the $R^2$ score of the model's predictions
        - Also draws a plot of the raw data and predictions
    '''
    model = LinearRegression()
    model.fit(X_proc, y)

    if show:
        fig = plot_raw_data_and_predictions(X_raw, X_proc, y, model)
        fig.show(renderer='notebook')

    params = np.append(model.intercept_, model.coef_).flatten()
    rmse = np.sqrt(mean_squared_error(y, model.predict(X_proc)))
    r2 = model.score(X_proc, y)

    return (params, rmse, r2)

# Start with this as a template.
def create_and_train_1a(X_raw, y, show=True):
    """
    Inputs:
        - X_raw, a DataFrame with a single column, `Year`
        - y, a Series with `Homeruns` values
    Outputs:
        - Returns a tuple containing the model's optimal parameters, the root mean squared error of the model's predictions, and the $R^2$ score of the model's predictions
        - Also draws a plot of the raw data and predictions
    """
    # Create new features
    X_proc = X_raw.copy() # Don't forget this; otherwise, you may make in-place modifications to the raw data.
    X_proc['Year'] = X_proc['Year'] # Already there; just including this line to show that it hasn't been removed.
    X_proc['Year^2'] = X_proc['Year'] ** 2

    return create_and_train_helper(X_raw, X_proc, y, show=show)
create_and_train_1a(homeruns[['Year']], homeruns['Homeruns'])


Given the above information, the **fit** model is:

$$h^*(x_i) = 769420.7 - 829.9x_i + 0.2237 x_i^2$$

Again, it's not required that you write this formula down, but you should be able to.

In [ ]:
grader.check("q01a")

### Problem 1b) (4 pts) [Autograded 💻]

$$h(x_i) = w_0 + w_1 x_i + w_2 x_i^2 + w_3 x_i^3 + w_4 x_i^4$$

In [ ]:
def create_and_train_1b(X_raw, y, show=True):
    ...

create_and_train_1b(homeruns[['Year']], homeruns['Homeruns'])

In [ ]:
grader.check("q01b")

### Problem 1c) (4 pts) [Autograded 💻]

$$h(x_i) = w_0 + w_1 x_i + w_2 (x_i - 1900) \cos\left(\frac{2 \pi}{25} x_i \right) + w_3 e^{\frac{x_i - 1900}{100}}$$

In [ ]:
def create_and_train_1c(X_raw, y, show=True):
    ...

create_and_train_1c(homeruns[['Year']], homeruns['Homeruns'])

In [ ]:
grader.check("q01c")

<!-- BEGIN QUESTION -->

### Problem 1d) (3 pts) [Manually Graded ✏️]
Why can’t we use linear regression to fit the model  
$$
h(x_i) = w_0 + w_1 x_i + w_2 (x_i - 1900)\cos(w_3 x_i) + w_4 e^{\frac{x_i - 1900}{w_5}}?
$$

In the cell below, write a 2-3 sentence explanation.

<hr style="border: 0; height: 4px; background: royalblue;">

_Your answer goes here, replacing this text._

<hr style="border: 0; height: 4px; background: royalblue;">

<!-- END QUESTION -->

---

## Problem 2: Polynomial Regression (12 pts)

In this problem, we'll explore the implications of adding more and more complex features to a model.


<!-- BEGIN QUESTION -->

### Part 2a) (4 pts) [Manually Graded ✏️]

Suppose $\vec y \in \mathbb{R}^n$ is an observation vector, $X$ is an $n \times (d+1)$ design matrix, and $X_{\text{new}}$ is an $n \times (d+2)$ design matrix whose first $d+1$ columns are identical to those of $X$, while its last column contains one additional feature.

$$
X_{\text{new}} =
\begin{bmatrix}
& | \\
X & \vec x^{(d+1)} \\
& |
\end{bmatrix}
$$

These design matrices correspond to models $h(\vec x_i)$ and $h_\text{new}(\vec x_i)$, where $h_\text{new}(\vec x_i)$ includes one additional feature.

Let $\vec w^* \in \mathbb{R}^{d+1}$ be a minimizer of

$$
\frac{1}{n}\left\lVert \vec y - X\vec w \right\rVert^2
$$

and let $\vec w^*_\text{new} \in \mathbb{R}^{d+2}$ be a minimizer of

$$
\frac{1}{n}\left\lVert \vec y - X_{\text{new}}\vec w_\text{new} \right\rVert^2.
$$

Note that $\vec w^*_\text{new}$ has one more component than $\vec w^*$. In general, the components of $\vec w^*$ and $\vec w^*_\text{new}$ do not have a direct relationship: adding a new feature can change the optimal coefficients for all of the existing features. 

In 1-2 English sentences, **explain why**

$$
\frac{1}{n}\left\lVert \vec y - X_\text{new}\vec w^*_\text{new} \right\rVert^2
\leq
\frac{1}{n}\left\lVert \vec y - X\vec w^* \right\rVert^2.
$$

This inequality is equivalent to the statement "adding a feature never increases the mean squared error of the model’s predictions on the data it was trained on." You don't need to do much math to answer this.

<hr style="border: 0; height: 4px; background: royalblue;">

_Your answer goes here, replacing this text._

<hr style="border: 0; height: 4px; background: royalblue;">

<!-- END QUESTION -->

Now, run the cell below to generate the dataset for this problem. **Do not** modify the code in this cell!

In [ ]:
np.random.seed(23) # For reproducibility.

def sample_from_pop(n):
    x = np.linspace(-3, 3, n)
    y = (x**3) + (np.random.normal(0, 5, size=n))
    return pd.DataFrame({'x': x, 'y': y})

full = sample_from_pop(n=200)
full.plot(kind='scatter', x='x', y='y')

In the previous part, we had you think about how adding a new feature impacts the model's **training** mean squared error, i.e. its performance on the $n$ data points it was trained on. But, adding new features can hurt a model's performance on unseen data, often called **test** data, which is ultimately the performance we care about. So, to assess a model's ability to **generalize** to unseen data, we typically split our data into a training set and test set, fit the model on just the training set, and then evaluate its performance on the test set.

<p align="center">
  <img src="../imgs/train-test-first.png" width="50%">
</p>

Here's the general idea we'll explore. Suppose we want to fit a polynomial regression model of degree $d$, i.e. a model of the form

$$h(x_i) = w_0 + w_1 x_i + w_2 x_i^2 + ... + w_d x_i^d$$

where $x_i$ is a scalar. **The question is, what value of $d$ should we choose?** As $d$ increases, the model's training mean squared error will stay the same or decrease. But, its test mean squared error may increase.

Let's see this in action. First, a train-test split.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and test sets.
# The random_state ensures that we get the same "split" of the data each time we run this cell.
X_train_poly, X_test_poly, y_train_poly, y_test_poly = train_test_split(full[['x']], full['y'], test_size=0.2, random_state=15)

In [ ]:
X_train_poly, y_train_poly

In [ ]:
X_test_poly, y_test_poly

Instead of creating the features manually, like in Problem 1, we'll use some more advanced features in `sklearn` to create the features for us.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_squared_error

Below, we create a **Pipeline** that first creates polynomial features of degree **20** (chosen as an example), and then fits a `LinearRegression` model on the transformed data. Pay close attention to this code, as you'll need to use it in your implementation of part b).

In [ ]:
model = make_pipeline(PolynomialFeatures(degree=20, include_bias=False), LinearRegression())
model

In [ ]:
model.fit(X_train_poly, y_train_poly)

The model's training set and test set mean squared errors are below. (Unlike in Problem 1, we're using MSE instead of RMSE since the $y$-axis scale is already relatively small.)

In [ ]:
train_mse = mean_squared_error(y_train_poly, model.predict(X_train_poly))
test_mse = mean_squared_error(y_test_poly, model.predict(X_test_poly))
print(f"Training MSE: {train_mse:.2f}")
print(f"Test MSE: {test_mse:.2f}")

Note that the model performed significantly worse on the test set. Let's look at the model overlaid on both datasets.

In [ ]:
import numpy as np
import plotly.graph_objs as go
from plotly.subplots import make_subplots

def plot_model_train_test(model, X_train, y_train, X_test, y_test):
    # Make prediction domain, but clip xs to [-2, 2.5]
    xs = np.linspace(-3, 3, 300).reshape(-1, 1)
    y_pred = model.predict(xs)
    
    # Set up figure with 1 row, 2 columns
    train_mse = mean_squared_error(y_train, model.predict(X_train))
    test_mse = mean_squared_error(y_test, model.predict(X_test))
    degree = model[0].degree
    fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Training Data (MSE for degree {degree}: {train_mse:.2f})", f"Test Data (MSE for degree {degree}: {test_mse:.2f})"))

    # (1) Left: training data
    fig.add_trace(
        go.Scatter(x=X_train.squeeze(), y=y_train, mode='markers', name="Train data"),
        row=1, col=1
    )
    # Model prediction curve (orange)
    fig.add_trace(
        go.Scatter(x=xs.squeeze(), y=y_pred, mode='lines', name="Predictions", line=dict(color='orange', width=4)),
        row=1, col=1
    )
    
    # (2) Right: test data
    fig.add_trace(
        go.Scatter(x=X_test.squeeze(), y=y_test, mode='markers', name="Test data"),
        row=1, col=2
    )
    # Model prediction curve (orange)
    fig.add_trace(
        go.Scatter(x=xs.squeeze(), y=y_pred, mode='lines', name="Predictions", showlegend=False, line=dict(color='orange', width=4)),
        row=1, col=2
    )
    
    # Clip x-axes to [-2, 2.5] in both subplots
    fig.update_xaxes(title_text="X", range=[-3, 3], row=1, col=1)
    fig.update_yaxes(title_text="y", row=1, col=1)
    fig.update_xaxes(title_text="X", range=[-3, 3], row=1, col=2)
    fig.update_yaxes(title_text="y", row=1, col=2)

    fig.update_layout(height=400, width=900, showlegend=True)
    fig.show(renderer='notebook')

plot_model_train_test(model, X_train_poly, y_train_poly, X_test_poly, y_test_poly)

Note that the model performed significantly worse on the test set. This, at some level, is expected; the model got to see the training data when choosing $\vec w^*$, and so it may have fit the training data too closely. But is 20 the best degree?

### Part 2b) (5 pts) [Autograded 💻]

Now it's your turn to complete the exploration. Below, complete the implementation of the function `plot_train_and_test_vs_degree`. The function should take in four arguments – `X_train_poly`, `y_train_poly`, `X_test_poly`, and `y_test_poly` – and should:

1. Fit 25 polynomial regression models on the training set (degree 1, degree 2, ..., degree 25).
1. For each model compute both its training and test mean squared error.
1. Return a line plot of the training and test mean squared error vs. degree. (There is helper code for this below.)

This function is autograded; we will be programatically checking that the plot that `plot_train_and_test_vs_degree` returns has the correct values.

In [ ]:
# Helper function for plotting the training and test mean squared error vs. degree.
# Requires three lists/arrays of the same length: `deg`, `train_mses`, `test_mses`.
def plot_two_lines(deg, train_mses, test_mses):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=deg, y=train_mses, mode='lines+markers', name='Train MSE'))
    fig.add_trace(go.Scatter(x=deg, y=test_mses, mode='lines+markers', name='Test MSE'))
    fig.update_layout(
        title="Training and Test MSE vs. Degree",
        xaxis_title='Degree',
        yaxis_title='MSE'
    )
    return fig

def plot_train_and_test_vs_degree(X_train_poly, y_train_poly, X_test_poly, y_test_poly):
    ...

plot_train_and_test_vs_degree(X_train_poly, y_train_poly, X_test_poly, y_test_poly)

In [ ]:
grader.check("q02b")

<!-- BEGIN QUESTION -->

### Part 2c) (3 pts) [Manually Graded ✏️]

You'll have noticed that the choice of $d$ (degree) with the smallest **test** mean squared error is a relatively small number. (There will be a few values of $d$ with very similar test mean squared errors, but hover over your plot: there is a unique minimum.)

In the cell below, in 1-3 English sentences, explain **why** adding more and more features (i.e. increasing $d$) leads to worse model performance on the test set, and what lesson we can take from this in the context of building (not necessarily just polynomial) regression models that generalize to unseen data.

<hr style="border: 0; height: 4px; background: royalblue;">

_Your answer goes here, replacing this text._

<hr style="border: 0; height: 4px; background: royalblue;">

<!-- END QUESTION -->

If you'd like to learn more about the core ideas explored in this problem, consult [EECS 398: Practical Data Science](https://practicaldsc.org/).

---

## Problem 3: One Hot Encoding in `sklearn` (10 pts)

In [Chapter 7.2](https://notes.eecs245.org/regression-using-linear-algebra/multiple-linear-regression/#example-one-hot-encoding), we showed you how to one hot encode a categorical feature **manually** in `pandas`. In general, we will want to use `sklearn`'s `OneHotEncoder` class to one hot encode a categorical feature.

In this problem, we'll aim to build a regression model that predicts the price of a house based on various features about it. The dataset we're using, originally compiled by Professor Dean De Cock at Truman State University **specifically for** teaching regression, contains information about houses sold in Ames, Iowa from 2006 to 2010.

Run the cell below to load in the data. A full data dictionary can be found [here](https://www.openintro.org/data/index.php?data=ames).

In [ ]:
houses = pd.read_csv('data/iowa.csv')
houses

Each row corresponds to a house. There are 80 columns (so 79 possible features); the last column contains the target variable, `SalePrice`.

In [ ]:
houses.columns

In [ ]:
fig = houses.plot(kind='scatter', x='Gr Liv Area', y='SalePrice')
fig.update_layout(title='Square Footage (Excluding Basement) vs. Sale Price')
fig.show(renderer='notebook')

In [ ]:
fig = houses['Neighborhood'].value_counts().plot(kind='bar')
fig.update_layout(title='Distribution of Neighborhoods')
fig.show(renderer='notebook')

As we learned in Problem 2, we should always perform a train/test split before building a model. Let's do that now.

In [ ]:
X_train_houses, X_test_houses, y_train_houses, y_test_houses = train_test_split(houses.drop(columns=['SalePrice']), houses['SalePrice'], test_size=0.2, random_state=15)

In Problem 2, we looked at how to create an `sklearn` Pipeline that first creates polynomial features using the `PolynomialFeatures` class, and then fits a `LinearRegression` model on the transformed data. But there, we wanted to create polynomial features out of every single "original" feature, which was just the $x$-variable. But here, it might make sense to use `Gr Liv Area` (non-basement square footage) as a numerical feature as-is, and then one hot encode categorical features, like `Neighborhood`.

The big idea is that we need a way to tell `sklearn` what operations to apply on each feature. The solution is to use a `ColumnTransformer` object, which allows us to specify different operations for different columns. Let's look at an example of how this works.

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer

model = make_pipeline(
    make_column_transformer(
        (OneHotEncoder(drop='first', handle_unknown='ignore'), ['Neighborhood']),
        (FunctionTransformer(lambda x: x, validate=False), ['Gr Liv Area']),
        remainder='drop'
    ),
    LinearRegression()
)

model

In [ ]:
model.fit(X_train_houses, y_train_houses)

You might have notice that after fitting the model, it appears blue (the same happened in the previous problem, too).

A lot of syntax went into defining `model`. Let's break it down:
1. First, `model` itself is a Pipeline. Its first step is a `ColumnTransformer`, which is the "controller" that tells `sklearn` what to do with each column. The second step is a `LinearRegression` model.
1. The `ColumnTransformer`, created using `make_column_transformer`, is instantiated using several tuples, each one containing the name of a transformation and a list of relevant columns.
    - We said apply `OneHotEncoder(drop='first', handle_unknown='ignore')` to the `Neighborhood` column.
    - `FunctionTransformer(lambda x: x, validate=False)` looks a little strange, but all it's saying is to **keep `Gr Liv Area` as is**.
    - `remainder='drop'` tells `sklearn` to drop the columns that are not mentioned in the list of tuples.
1. The `LinearRegression` model is the usual `LinearRegression` model.

We can peek into how the model was fit and transformed our data. First, we can look at its coefficients.

In [ ]:
# Access step 2 of the Pipeline, which is the LinearRegression model.
model[1]

In [ ]:
model[1].coef_

In [ ]:
# Total number of parameters in the model (including the intercept term, which is not accounted for above.)
len(model[1].coef_) + 1

Many of the coefficients above correspond to one hot encoded features. Let's look at the names of the one hot encoded features.

In [ ]:
model[0]

In [ ]:
model[0]['onehotencoder'].get_feature_names_out()

How many one hot encoded features did the model create?

In [ ]:
len(model[0]['onehotencoder'].get_feature_names_out())

But, in `X_train_houses`, how many unique `Neighborhood` values are there?

In [ ]:
X_train_houses['Neighborhood'].nunique()

Hmmm... what gives? The reason that our model produced one fewer one hot encoded features than there were unique `Neighborhood` values is that we told `sklearn` to drop one category of `Neighborhood` (the first one that it saw, to be exact). We did this to ensure that the resulting model's design matrix was of full column rank, meaning that all of its columns are linearly independent. As we've discussed in [Chapter 7.2](https://notes.eecs245.org/regression-using-linear-algebra/multiple-linear-regression/#example-one-hot-encoding), this ensures that we have a unique solution for $\vec w^*$, our optimal parameter vector.

The cool thing is that to use our new model to make predictions, we can just use the usual `.predict` method, and don't need to worry about re-implementing the logic of one hot encoding.

For example, let's predict how much a house with 2550 square feet of living area in the `'CollgCr'` neighborhood is estimated to sell for.

In [ ]:
model.predict(pd.DataFrame({'Gr Liv Area': [2550], 'Neighborhood': ['CollgCr']}))

Just under \$285,000, it seems!


What if we make up a new neighborhood, like `'Ann Arbor'`, which definitely doesn't exist in the dataset?

In [ ]:
model.predict(pd.DataFrame({'Gr Liv Area': [2550], 'Neighborhood': ['Ann Arbor']}))

You **might** get a `UserWarning`, but not an error. We avoided any errors because we instantiated `OneHotEncoder` with `handle_unknown='ignore'`, which tells it to ignore any `Neighborhood`categories it sees in `.predict` that it didn't see in `.fit`.

What is this model's training and test mean squared error?

In [ ]:
train_mse = mean_squared_error(y_train_houses, model.predict(X_train_houses))
test_mse = mean_squared_error(y_test_houses, model.predict(X_test_houses))
print(f"Training MSE: {train_mse:.2f}")
print(f"Test MSE: {test_mse:.2f}")

### Problem 3a) (7 pts) [Autograded 💻]

Below, complete the implementation of the function `house_pipeline`, which takes in two arguments – `X_train_houses`, `y_train_houses` – and should:

1. Create `PolynomialFeatures` of degree 3 for the `Gr Liv Area` column (make sure to set `include_bias=False`, since `LinearRegression` will add its own intercept term).
2. Use the `Yr Sold` and `Year Built` columns as-is.
3. One hot encode **both** the `Neighborhood` column and the `MS SubClass` columns.
4. Train a `LinearRegression` model on the transformed **training** data, ignoring/dropping any other columns.
5. Return the fit Pipeline object.

In [ ]:
...

    model_big.fit(X_train_houses, y_train_houses)
    return model_big

house_pipeline(X_train_houses, y_train_houses)

In [ ]:
grader.check("q03a")

### Problem 3b) (3 pts) [Autograded 💻]

Below, complete the implementation of the function `house_pipeline_metrics`, which takes in four arguments – `X_train_houses`, `y_train_houses`, `X_test_houses`, and `y_test_houses` – and should:

1. Use the `house_pipeline` function to fit a model on the training data.
1. Return a tuple containing:
    1. The model's training mean squared error.
    2. The model's test mean squared error.
    3. The **number of parameters** in the model.

In [ ]:
def house_pipeline_metrics(X_train_houses, y_train_houses, X_test_houses, y_test_houses):
    ...

house_pipeline_metrics(X_train_houses, y_train_houses, X_test_houses, y_test_houses)

In [ ]:
grader.check("q03b")

---

## Problem 4: Implementing Gradient Descent (12 pts)

In this problem, you'll implement gradient descent yourself, to find optimal model parameters for a simple linear regression model.

To set up the problem, let's load in the now-familiar commute times dataset from lecture.

In [ ]:
df = pd.read_csv('data/commute-times.csv')
df.head()

One of our running examples has been to build a simple linear regression model of the form $h(x_i) = w_0 + w_1 x_i$, that predicts commute time in `minutes` given `departure_hour`.

In [ ]:
line_fig = px.scatter(df, x='departure_hour', y='minutes').update_layout(xaxis_title='Home Departure Time (AM)', yaxis_title='Minutes', title='Commuting Time vs. Home Departure Time')
line_fig.show(renderer='notebook')

The "default" approach has been to choose squared loss, meaning that we choose the intercept, $w_0^*$, and slope, $w_1^*$, that minimize **mean squared error**:

$$R_\text{sq}(w_0, w_1) = \frac{1}{n} \sum_{i = 1}^n \left( y_i - (w_0 + w_1 x_i) \right)^2$$

We solved for $w_0^*$ and $w_1^*$ by hand using calculus in [Chapter 2.3](https://notes.eecs245.org/simple-linear-regression/finding-optimal-parameters/), and in [Chapter 6.3](https://notes.eecs245.org/linear-transformations-and-projections/projecting-onto-column-space/) and [Chapter 7.1](https://notes.eecs245.org/regression-using-linear-algebra/regression-using-linear-algebra/) we found an equivalent solution using linear algebra (which is how we found the normal equation). We could also use gradient descent to minimize $R_\text{sq}(w_0, w_1)$.

For reference, we'll calculate the slope and intercept that minimize mean squared error below.

In [ ]:
from sklearn.linear_model import LinearRegression
baseline = LinearRegression()
baseline.fit(df[['departure_hour']], df['minutes'])
w_squared_loss = baseline.intercept_, baseline.coef_[0]
w_squared_loss

So, by minimizing mean squared error, we have $w_0^* = 142.45$ and $w_1^* = -8.19$. Keep these values in mind throughout the question. For reference, here's what <b><span style="color:orange">the line that minimizes mean squared error</span></b> looks like:

In [ ]:
line_fig.add_trace(
    go.Scatter(x=[5.5, 11.5], y=[baseline.predict([[5.5]])[0], baseline.predict([[11.5]])[0]], mode='lines', line=dict(color='orange'), name='Best Line (MSE)')
)

line_fig.show()

<br>

Another approach is to choose absolute loss, meaning that we choose intercept and slope that minimize **mean absolute error**:

$$R_\text{abs}(w_0, w_1) = \frac{1}{n} \sum_{i = 1}^n \left| y_i - (w_0 + w_1 x_i) \right|$$

We can't solve for the minimizer to $R_\text{abs}$ by-hand; the algebra doesn't work out. We also can't directly use gradient descent to minimize $R_\text{abs}$ since it's not differentiable. In an earlier homework, you implemented a brute-force computational routine that found the optimal slope and intercept in $O(n^3)$ time. 

<br>

What we'll explore here is a **new** loss function, Tukey's loss function, named after [John Tukey](https://en.wikipedia.org/wiki/John_Tukey), the creator of the box plot. It is defined as follows:

$$L_T(y_i, h(x_i)) = \begin{cases} 1 - \left( 1 - \left( \frac{y_i - h(x_i)}{50} \right)^2 \right)^3 && \text{if} \: |y_i - H(x_i)| \leq 50, \\ 1 && \text{otherwise} \end{cases}$$

To make sense of how the loss function behaves, let's graph it.

In [ ]:
def tukey(y_actual, y_pred, c=50):
    error = y_actual - y_pred
    if np.abs(error) <= c:
        return 1 - (1 - (error / c) ** 2) ** 3
    else:
        return 1

hs = np.linspace(-200, 200, 10000)
fig = px.line(x=hs, y=[tukey(0, h) for h in hs], title='Tukey Loss').update_layout(xaxis_title='h', yaxis_title='L(0, h)')
fig.show()

Note that Tukey loss is defined **piecewise**. 
- For predictions in which $|y_i - h(x_i)|$ is more than 50, the loss is capped at 1, resulting in a loss function that is _very_ robust to outliers (unlike squared loss, which is influenced heavily by outliers). 
- For predictions in which $|y_i - h(x_i)| \leq 50$, the loss looks very similar to squared loss. 

The choice of 50 as the threshold was arbitrary; we could have chosen a different transition point (and, once you finish the question, it's worth investigating how your results would have changed if 50 was replaced with 5 or 100).

You'll also notice that the "handoff" from the curved part to the flat part at $h = \pm 50$ is smooth, meaning Tukey loss is differentiable, unlike absolute loss. This will be important!

Let's continue to consider the simple linear regression model, $h(x_i) = w_0 + w_1 x_i$, where $x_i$ is a scalar, not a vector. In that case, the empirical risk $R_T$ using Tukey loss looks like:

$$R_T(w_0, w_1) = \frac{1}{n} \sum_{i = 1}^n  \begin{cases} 1 - \left( 1 - \left( \frac{y_i - (w_0 + w_1 x_i)}{50} \right)^2 \right)^3 && \text{if} \: |y_i - (w_0 + w_1 x_i)| \leq 50, \\ 1 && \text{otherwise} \end{cases}$$

Let's graph the loss surface, using some help from the helper functions we've defined in `hw09_util.py`. Note that we're specifying that we want the $x$'s and $y$'s to come from the `departure_hour` and `minutes` columns in `df`, respectively.

In [ ]:
fig = util.draw_loss_surface_tukey_commute(xs=df['departure_hour'], ys=df['minutes'])
fig.show()

Let's use gradient descent to find the intercept, $w_0^*$, and slope, $w_1^*$, that minimize the loss surface above. That is, let's find the best intercept and slope to use in a simple linear model that predicts `minutes` using `departure_hour`, using Tukey loss. We can think of this problem as trying to find the vector, $\vec w^*$, that minimizes $R_T(\vec w)$, where $\vec w = \begin{bmatrix} w_0 \\ w_1 \end{bmatrix}$.

The reason we're using gradient descent here is that $R_\text{T}$ can't be minimized by hand: you can find its gradient by hand, but solving for where the gradient is 0 can't be done by hand.

The gradient descent update rule is as follows:

$$\vec w^{(t+1)} = \vec w^{(t)} - \alpha \nabla R_T( \vec w^{(t)})$$

where $\vec w^{(t)}$ is our guess for the minimizing $\vec w^*$ at timestep $t$. To start the process, we'll need to decide on an initial guess, $\vec w^{(0)}$, and a step size, $\alpha$, which we will do later.

But, more crucially, to run gradient descent ourselves, we'll need to be able to compute $\nabla R_T(\vec w^{(t)})$, i.e. the **gradient** of $R_T$ at any point $\vec w$.

### Problem 4a) (7 pts) [Autograded 💻]

As a refresher, the empirical risk function that we're trying to minimize is:

$$R_T(\vec w) = R_T(w_0, w_1) = \frac{1}{n} \sum_{i = 1}^n  \begin{cases} 1 - \left( 1 - \left( \frac{y_i - (w_0 + w_1 x_i)}{50} \right)^2 \right)^3 && \text{if} \: |y_i - (w_0 + w_1 x_i)| \leq 50, \\ 1 && \text{otherwise} \end{cases}$$

Complete the implementation of the function `tukey_gradient`, which takes in:
- `w`, an **array of length 2**, corresponding to a value of $w_0$ and a value of $w_1$, and
- `X_train_commutes` and `y_train_commutes`, two Series/1D arrays with the same length, corresponding to sequences of $x$-values (like `df['departure_hour']`) and $y$-values (like `df['minutes']`), respectively.

`tukey_gradient` should return an **array of length 2**, containing the value of the gradient of $R_T$, evaluated at the point `w` that is passed in. Example behavior is given below.

```python
# This says that dR/dw0, when (w0, w1) = (100, -5), is -0.02389409, and
# dR/dw1, when (w0, w1) = (100, -5), is -0.20369193.
>>> tukey_gradient(np.array([100, -5]), X_train_commutes=df['departure_hour'], y_train_commutes=df['minutes'])
array([-0.02389409, -0.20369193])
```

Some guidance:
- Remember, the gradient of a function is a vector of partial derivatives. So, $\nabla R_T(\vec w) = \begin{bmatrix} \frac{\partial R}{\partial w_0} \\ \frac{\partial R}{\partial w_1} \end{bmatrix}$.
- Because $R_T$ is a piecewise function, both partial derivatives will also be piecewise, using the same condition as in $R_T$. So, your definition of `tukey_gradient` can involve `if`-statements.
- You'll need to do a substantial amount of math on-paper to complete this implementation, and accurately translate it to code. `for`-loops are fine in your implementation.
    - Back in [Chapter 2.3](https://notes.eecs245.org/simple-linear-regression/finding-optimal-parameters/) (which you should review for this!), we stressed that the derivative of a sum of functions is equal to the sum of the derivatives of those functions. In other words, $\frac{\partial R_T}{\partial w_0} = \frac{1}{n} \sum_{i = 1}^N \frac{\partial L_T}{\partial w_0}$. Given this, it's easiest to start with finding the partial derivatives of just the loss function $L_T$ with respect to $w_0$ and $w_1$, and summing these partial derivatives in a `for`-loop.
    - When computing the partial derivatives of $L_T$ with respect to the two parameters, you may find it helpful to perform a substitution, e.g. $e_i = y_i - (w_0 + w_1 x_i)$, and then use the chain rule for scalar-to-scalar functions, i.e. $\frac{\partial L_T}{\partial w_0} = \frac{\partial L_T}{\partial e_i} \cdot \frac{\partial e_i}{\partial w_0}$.
    - $\frac{\partial R}{\partial w_0}$ and $\frac{\partial R}{\partial w_1}$ will look very similar, but with one small difference.

In [ ]:
def tukey_gradient(w, X_train_commutes, y_train_commutes):
    ...

# Feel free to change these inputs to make sure your functions work correctly.
tukey_gradient(np.array([100, -5]), X_train_commutes=df['departure_hour'], y_train_commutes=df['minutes'])

In [ ]:
grader.check("p04a")

### Problem 4b) (5 pts) [Autograded 💻]

Complete the implementation of the function `run_gradient_descent_tukey`, which takes in:
- `w_initial`, an **array of length 2**, corresponding to an initial guess, $\vec w^{(0)}$, of the minimizer.
- `alpha`, a positive number corresponding to a step size/learning rate.
- `X_train_commutes` and `y_train_commutes`, two Series/1D arrays with the same length, corresponding to sequences of $x$-values (like `df['departure_hour']`) and $y$-values (like `df['minutes']`), respectively.
- `tol`, a float representing the **convergence criteria**, the default value of which will be set to `0.0001` (i.e. $10^{-3}$).
- `verbose`, a Boolean flag.

`run_gradient_descent_tukey` should run as many iterations of gradient descent as necessary, terminating once $\lVert \nabla R_T(\vec w^{(t)}) \rVert_2 \leq \text{tol}$, i.e. once the $L_2$ norm of `tukey_gradient(w, X_train_commutes, y_train_commutes)` (where `w` is the current guess of $\vec w^*$) is less than `tol`.
`run_gradient_descent_tukey` should return a 2D array containing all vectors $\vec w^{(t)}$ that were visited by gradient descent. In other words, it should return a 2D array of shape `(num_iterations, 2)`, where row 0 is $\vec w^{(0)}$ (the initial guess), row 1 is $\vec w^{(1)}$, row 2 is $\vec w^{(2)}$, and so on, until row -1 is our final guess of $\vec w^*$.

If `verbose=True`, then on iterations 0, 1000, 2000, and so on, use the `display` function to show the iteration number $t$, the value of $\vec w^{(t)}$, the value of  $R_T(\vec w^{(t)})$, and the value of $\lVert \nabla R_T(\vec w^{(t)}) \rVert_2$ (i.e. the norm of the gradient vector) at the current iteration. You can compute $R_T(\vec w^{(t)})$ using the `tukey` function defined at the top of the notebook, or using `util.empirical_risk`. We won't test your code with `verbose=True`, so you have some flexibility in how to implement it, but you'll need to complete this step in order for the remainder of the problem to make sense.

Finally, if more than 50,000 iterations have been completed (including the initial guess), terminate the algorithm and return an array of shape `(50000, 2)` containing the 50,000 vectors $\vec w^{(0)}, \vec w^{(1)}, ... \vec w^{(49,999)}$ that were visited by the algorithm.

Example behavior is given below.

```python
>>> path = run_gradient_descent_tukey(w_initial=np.array([100, 0]), 
                                      alpha=10, X_train_commutes=df['departure_hour'], y_train_commutes=df['minutes'])

# Our guess of w^*, the optimal parameter vector.
>>> path[-1]
array([121.5414978 ,  -5.85456359])

# The number of steps it took to reach the above guess, including the first and last guesses.
>>> len(path)
7616
```

In [ ]:
def run_gradient_descent_tukey(w_initial, alpha, X_train_commutes, y_train_commutes, tol=0.0001, verbose=False):
    ...

# Feel free to change these inputs to make sure your functions work correctly.
path = run_gradient_descent_tukey(w_initial=np.array([100, 0]), 
                           alpha=10, 
                           X_train_commutes=df['departure_hour'], 
                           y_train_commutes=df['minutes'], 
                           verbose=True)
path[-1]

In [ ]:
grader.check("p04b")

Nice work! Let's experiment with what you've done. The expression below will call your implementation of `run_gradient_descent_tukey`, and draw out the path that gradient descent took to minimize $R_T(\vec w)$ on the loss surface of $R_T(\vec w)$ itself, using an initial guess of $\vec w^{(0)} = \begin{bmatrix} 100 \\ 0 \end{bmatrix}$ and $\alpha = 10$ (as in the example output provided at the start of part b)). If you hover over a point in gold, you'll see its iteration number, $t$. (Scroll to the right to see the contour plot.)

In [ ]:
path = run_gradient_descent_tukey(w_initial=np.array([100, 0]), alpha=10, X_train_commutes=df['departure_hour'], y_train_commutes=df['minutes'])
fig = util.draw_loss_surface_tukey_commute(xs=df['departure_hour'], ys=df['minutes'], path=path)
fig.show()

You should notice that within a few steps, gradient descent gets down to the valley, but it takes thousands more iterations to inch sufficiently close to the true minimum. If you called `run_gradient_descent_tukey` with `verbose=True` above, you should have seen that the value of $R_T(\vec w^{(t)})$ decreased very slowly every 1000 iterations, and the norm of the gradient vector very, very slowly approached 0. If we settled for a greater tolerance – say, $0.0005$ instead of $0.0001$, we'd have converged quicker:

In [ ]:
run_gradient_descent_tukey(w_initial=np.array([100, 0]), 
                           alpha=10, 
                           X_train_commutes=df['departure_hour'], 
                           y_train_commutes=df['minutes'], 
                           tol=0.0005,
                           verbose=True)

But, the resulting $\vec w^*$ of $\begin{bmatrix} 105.234 \\ -3.978 \end{bmatrix}$ is quite far from what we got when using a tolerance of $0.0001$, which gave us $\begin{bmatrix} 121.541 \\ -5.855 \end{bmatrix}$. Why do you think this is happening?

Instead of weakening our tolerance, perhaps we can try a different learning rate:

In [ ]:
run_gradient_descent_tukey(w_initial=np.array([100, 0]), 
                           alpha=15, 
                           X_train_commutes=df['departure_hour'], 
                           y_train_commutes=df['minutes'], 
                           tol=0.0001,
                           verbose=True)

If you've implemented everything correctly, you should see that the above call to gradient descent maxes out at 50,000 iterations. Something strange must be going on, since the guesses of $\vec w^{(t)}$ seem to be relatively constant every 1000 iterations, as do the values of $R_T(\vec w^{(t)})$ and $\lVert \nabla R_T(\vec w^{(t)}) \rVert_2$. What's going on? Make an educated guess, then run the cell below. (It should take ~30 seconds to render.)

In [ ]:
path = run_gradient_descent_tukey(w_initial=np.array([100, 0]), alpha=15, X_train_commutes=df['departure_hour'], y_train_commutes=df['minutes'])
fig = util.draw_loss_surface_tukey_commute(xs=df['departure_hour'], ys=df['minutes'], path=path)
fig.show()

It seems that a learning rate of $\alpha = 15$ is too large, and results in oscillatory behavior between iterations $t$ and $t+1$. Since we're only printing every 1000 iterations when `verbose=True`, we only get to see one of the two oscillatory states (e.g., in the sequence $a$, $b$, $a$, $b$, $a$, $b$, ..., if we sample just the even positions, we'd think the entire sequence was made up of $b$'s).

Maybe an even larger learning rate will work better – let's try $\alpha = 50$.

In [ ]:
run_gradient_descent_tukey(w_initial=np.array([100, 0]), 
                           alpha=50, 
                           X_train_commutes=df['departure_hour'], 
                           y_train_commutes=df['minutes'], 
                           tol=0.0001,
                           verbose=True)

That was quick... what happened?

In [ ]:
path = run_gradient_descent_tukey(w_initial=np.array([100, 0]), alpha=50, X_train_commutes=df['departure_hour'], y_train_commutes=df['minutes'])
fig = util.draw_loss_surface_tukey_commute(xs=df['departure_hour'], ys=df['minutes'], path=path)
fig.show()

Why did gradient descent get stuck up top? Something you may have noticed is that $R_T(\vec w)$ is **not convex**. This means that there can be regions in which the gradient of $R_T(\vec w)$ is 0 that don't correspond to a global minimum (which isn't possible for convex functions). The step size of $\alpha = 50$ is so large that, after a few iterations, gradient descent lands us in the flat region, and once a $\vec w^{(t)}$ lands there, $\nabla R_T(\vec w^{(t)}) = \begin{bmatrix} 0 \\ 0 \end{bmatrix}$, terminating the algorithm instantly.

So, to summarize:
- Due to the nature of the loss surface, gradient descent can get _near_ the minimum fairly quickly, but may take many iterations to actually converge.
- If we choose the step size to be too large, gradient descent may oscillate infinitely, "bouncing" over the minimum.
- Since the loss surface is not convex, gradient descent can get "trapped" in flat regions and terminate mistakenly.

Hopefully, you now have a better understanding of how gradient descent works and how it can go wrong.

Finally, let's actually take a look 👀 at the line that minimizes mean Tukey loss on the commute times dataset.

In [ ]:
w0, w1 = run_gradient_descent_tukey(w_initial=np.array([100, 0]), 
                                    alpha=10, X_train_commutes=df['departure_hour'], y_train_commutes=df['minutes'],
                                    verbose=True)[-1]
w0, w1

In [ ]:
line_fig.add_trace(
    go.Scatter(x=[5.5, 11.5], y=[w0 + w1 * 5.5, w0 + w1 * 11.5], mode='lines', line=dict(color='purple'), name='Best Line (Mean Tukey Loss)')
)

line_fig.show()

Knowing what you know about Tukey loss, how does the <b><span style="color:orange">line that minimizes mean squared error</span></b> compare to the <b><span style="color: purple">line that minimizes mean Tukey loss</span></b>? You don't need to write the answer to this question – or any of the other thought-provoking questions in this notebook – anywhere, but you _should_ think about them.

## Finish Line 🏁

To get credit for **Homework 9B**, submit this notebook to Gradescope. There is **no corresponding PDF submission** for this notebook.

1. Select `Kernel -> Restart & Run All` to ensure that you have executed all cells, including the test cells.
2. Read through the notebook to make sure everything looks correct and that all public tests have passed. **Remember that there are no hidden tests in this notebook.**
3. Run the cell below to run all tests, and make sure they all pass.
4. Download your notebook using `File -> Download`, then upload it to Gradescope under **"Homework 9B Notebook"**.
5. After submitting, wait a few minutes for the Gradescope autograder to finish. The questions marked [Autograded 💻] will be **autograded on Gradescope**, while the questions marked [Manually Graded 📝] will be **manually graded**.